# PPE YOLO26 Kaggle Experiment Runner

This notebook runs ablation-friendly PPE violation detection experiments on Kaggle, including baseline YOLO26n, CBAM, GhostConv, violation-focused augmentation, and violation-class oversampling. It is designed for Kaggle T4 x2 but falls back to one GPU or CPU smoke checks when needed.

The comparison focus is violation detection for class IDs `7-10`: `no_helmet`, `no_goggle`, `no_gloves`, and `no_boots`. All experiments use the same dataset, split, seed, and evaluation code unless explicitly controlled by the variables below.

## 1. Control Variables

Edit this cell first. For a quick Kaggle sanity pass, keep `FAST_DEBUG=True` and select one or two experiments. For full runs, set `FAST_DEBUG=False` and choose the exact experiments you want in `RUN_EXPERIMENTS`.

In [ ]:
from pathlib import Path

# Kaggle paths. Leave DATASET_ROOT / PROJECT_ROOT as None to auto-detect.
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
DATASET_ROOT = None
PROJECT_ROOT = None
OUTPUT_ROOT = KAGGLE_WORKING / 'ppe_yolo_experiments'

# DEVICE is auto-filled later when set to None.
DEVICE = None

# Select any subset of these experiments:
# baseline_yolo26n, baseline_violation_aug, baseline_violation_oversample,
# cbam_existing_repro, cbam_violation_oversample,
# ghost_existing_repro, ghost_violation_oversample
RUN_EXPERIMENTS = [
    'baseline_yolo26n',
    'baseline_violation_aug',
    'baseline_violation_oversample',
    'cbam_existing_repro',
    'cbam_violation_oversample',
    'ghost_existing_repro',
    'ghost_violation_oversample',
]

# Runtime switches.
FAST_DEBUG = True
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_EXPORT = True

# Training controls.
EPOCHS = 50
DEBUG_EPOCHS = 1
IMGSZ = 640
DEBUG_IMGSZ = 320
BATCH = 64
SEED = 42

# Safety-critical violation classes.
VIOLATION_CLASS_IDS = [7, 8, 9, 10]
BEST_MODEL_SELECTION_METRIC = 'mean_violation_map50'

# Oversampling controls. These only generate manifest/list files in /kaggle/working.
OVERSAMPLE_FACTOR = 3
DEBUG_MAX_TRAIN_IMAGES = 128
DEBUG_MAX_EVAL_IMAGES = 64

print('Configured experiments:', RUN_EXPERIMENTS)
print('FAST_DEBUG:', FAST_DEBUG)

## 2. Imports, Output Directories, and Reproducibility

This section creates working directories, imports standard libraries, and fixes random seeds. It does not touch raw Kaggle input datasets.

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime

import numpy as np

OUTPUT_ROOT = Path(OUTPUT_ROOT)
RESULTS_DIR = KAGGLE_WORKING / 'results'
EXPORTS_DIR = KAGGLE_WORKING / 'exports'
LOGS_DIR = KAGGLE_WORKING / 'logs'
CONFIGS_DIR = KAGGLE_WORKING / 'configs'
MANIFESTS_DIR = KAGGLE_WORKING / 'manifests'
RUNS_DIR = OUTPUT_ROOT / 'runs'

for d in [OUTPUT_ROOT, RESULTS_DIR, EXPORTS_DIR, LOGS_DIR, CONFIGS_DIR, MANIFESTS_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('RESULTS_DIR:', RESULTS_DIR)
print('EXPORTS_DIR:', EXPORTS_DIR)

## 3. Inspect Kaggle Inputs

Kaggle datasets are mounted under `/kaggle/input`. This cell lists the tree up to depth 3 so you can confirm where the uploaded PPE dataset and project files are mounted.

In [ ]:
def list_tree(root: Path, max_depth: int = 3):
    root = Path(root)
    if not root.exists():
        print(f'Missing: {root}')
        return
    root = root.resolve()
    print(root)
    for path in sorted(root.rglob('*')):
        try:
            rel = path.relative_to(root)
        except ValueError:
            continue
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = '  ' * depth
        suffix = '/' if path.is_dir() else ''
        print(f'{indent}{path.name}{suffix}')

list_tree(KAGGLE_INPUT, max_depth=3)

## 4. Environment Check

This prints Python/package versions, GPU visibility, and CUDA device names. If two GPUs are available, training uses `device='0,1'`; if one GPU is available, training uses `device='0'`; otherwise CPU is used only for smoke/debug checks.

In [ ]:
def package_version(module_name, import_name=None):
    import_name = import_name or module_name
    try:
        mod = __import__(import_name)
        return getattr(mod, '__version__', 'installed-version-unknown')
    except Exception as exc:
        return f'not installed ({exc.__class__.__name__})'

print('Python:', sys.version)
for module_name, import_name in [
    ('torch', 'torch'),
    ('ultralytics', 'ultralytics'),
    ('numpy', 'numpy'),
    ('opencv', 'cv2'),
    ('onnx', 'onnx'),
    ('onnxruntime', 'onnxruntime'),
]:
    print(f'{module_name}: {package_version(module_name, import_name)}')

print('\n--- nvidia-smi ---')
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('nvidia-smi unavailable:', repr(exc))

try:
    import torch
    cuda_count = torch.cuda.device_count()
    print('torch.cuda.device_count():', cuda_count)
    for idx in range(cuda_count):
        print(f'GPU {idx}:', torch.cuda.get_device_name(idx))
    if DEVICE is None:
        DEVICE = '0,1' if cuda_count >= 2 else ('0' if cuda_count == 1 else 'cpu')
except Exception as exc:
    print('Torch CUDA inspection failed:', repr(exc))
    if DEVICE is None:
        DEVICE = 'cpu'

print('Selected DEVICE:', DEVICE)

## 5. Locate Project Files and Dataset Root

The notebook tries to find the dataset root containing `images/{train,val,test}` and `labels/{train,val,test}` automatically. It also finds the project root containing `ai-model/configs` when project files are mounted as a Kaggle dataset.

In [ ]:
def has_dataset_layout(path: Path) -> bool:
    path = Path(path)
    required = [
        path / 'images' / 'train', path / 'images' / 'val', path / 'images' / 'test',
        path / 'labels' / 'train', path / 'labels' / 'val', path / 'labels' / 'test',
    ]
    return all(p.exists() for p in required)

def find_dataset_root():
    if DATASET_ROOT is not None:
        return Path(DATASET_ROOT)
    candidates = []
    for base in [KAGGLE_INPUT, Path.cwd(), KAGGLE_WORKING]:
        if not base.exists():
            continue
        candidates.append(base)
        candidates.extend([p for p in base.rglob('*') if p.is_dir() and p.name.lower() in {'merged-ppe', 'dataset', 'ppe', 'data'}])
    for candidate in candidates:
        if has_dataset_layout(candidate):
            return candidate.resolve()
    # Fallback: any directory with both images and labels children.
    for candidate in candidates:
        if (candidate / 'images').exists() and (candidate / 'labels').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not auto-detect dataset root. Set DATASET_ROOT manually near the top of the notebook.')

def find_project_root():
    if PROJECT_ROOT is not None:
        return Path(PROJECT_ROOT)
    candidates = [Path.cwd(), KAGGLE_WORKING]
    if KAGGLE_INPUT.exists():
        candidates.extend([p for p in KAGGLE_INPUT.rglob('*') if p.is_dir()])
    for candidate in candidates:
        if (candidate / 'ai-model' / 'configs').exists():
            return candidate.resolve()
    return Path.cwd().resolve()

DATASET_ROOT = find_dataset_root()
PROJECT_ROOT = find_project_root()

print('DATASET_ROOT:', DATASET_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('Dataset layout detected:', has_dataset_layout(DATASET_ROOT))

## 6. Dataset Safety Checks

These checks verify that required split directories exist and that label files only contain valid class IDs. The checks read labels but do not modify the dataset.

In [ ]:
CLASS_NAMES = {
    0: 'helmet',
    1: 'gloves',
    2: 'vest',
    3: 'boots',
    4: 'goggles',
    5: 'none',
    6: 'Person',
    7: 'no_helmet',
    8: 'no_goggle',
    9: 'no_gloves',
    10: 'no_boots',
}
VALID_CLASS_IDS = set(CLASS_NAMES)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def split_dirs(root: Path, split: str):
    return root / 'images' / split, root / 'labels' / split

def image_files(img_dir: Path):
    return sorted([p for p in Path(img_dir).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def parse_label_classes(label_path: Path):
    classes = []
    if not label_path.exists():
        return classes
    text = label_path.read_text(errors='ignore').strip()
    if not text:
        return classes
    for line in text.splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        try:
            classes.append(int(parts[0]))
        except ValueError:
            classes.append(None)
    return classes

def validate_dataset(root: Path):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Dataset root not found: {root}')
    summary = {}
    invalid = []
    missing_labels = []
    for split in ['train', 'val', 'test']:
        img_dir, lbl_dir = split_dirs(root, split)
        if not img_dir.exists():
            raise FileNotFoundError(f'Missing image directory: {img_dir}')
        if not lbl_dir.exists():
            raise FileNotFoundError(f'Missing label directory: {lbl_dir}')
        imgs = image_files(img_dir)
        labels = sorted(lbl_dir.glob('*.txt'))
        counts = Counter()
        for img in imgs:
            label = lbl_dir / f'{img.stem}.txt'
            if not label.exists():
                missing_labels.append(str(label))
                continue
        for label in labels:
            for cls_id in parse_label_classes(label):
                if cls_id not in VALID_CLASS_IDS:
                    invalid.append((str(label), cls_id))
                else:
                    counts[cls_id] += 1
        summary[split] = {
            'images': len(imgs),
            'labels': len(labels),
            'class_counts': {CLASS_NAMES[i]: counts.get(i, 0) for i in sorted(CLASS_NAMES)},
        }
    if invalid:
        raise ValueError(f'Found invalid class IDs. First examples: {invalid[:10]}')
    if missing_labels:
        print(f'Warning: {len(missing_labels)} images are missing labels. First examples: {missing_labels[:10]}')
    return summary

dataset_summary = validate_dataset(DATASET_ROOT)
print(json.dumps(dataset_summary, indent=2))

## 7. Generate Kaggle Data YAML

Ultralytics infers labels from the matching `labels/{split}` directory when `train`, `val`, and `test` point to `images/{split}`. This YAML keeps the same class names as `data_modal.yaml` and is written under `/kaggle/working/configs/data_kaggle.yaml`.

In [ ]:
def write_data_yaml(path: Path, dataset_root: Path, train_value='images/train', val_value='images/val', test_value='images/test'):
    lines = [
        '# Kaggle-compatible dataset config generated by kaggle_ppe_experiments.ipynb',
        f'path: {dataset_root}',
        f'train: {train_value}',
        f'val: {val_value}',
        f'test: {test_value}',
        '# Label directories are inferred by Ultralytics as labels/train, labels/val, labels/test.',
        'names:',
    ]
    for idx, name in CLASS_NAMES.items():
        lines.append(f'  {idx}: {name}')
    path.write_text('\n'.join(lines) + '\n')
    return path

DATA_YAML = write_data_yaml(CONFIGS_DIR / 'data_kaggle.yaml', DATASET_ROOT)
print(DATA_YAML.read_text())

## 8. Model Config Checks

CBAM and Ghost experiments require the existing project YAML files. The baseline uses pretrained `yolo26n.pt` directly, matching the Modal baseline behavior.

In [ ]:
CBAM_YAML = PROJECT_ROOT / 'ai-model' / 'configs' / 'yolo26n-cbam.yaml'
GHOST_YAML = PROJECT_ROOT / 'ai-model' / 'configs' / 'yolo26n-ghost.yaml'

print('CBAM_YAML:', CBAM_YAML, 'exists=', CBAM_YAML.exists())
print('GHOST_YAML:', GHOST_YAML, 'exists=', GHOST_YAML.exists())

selected_requires_cbam = any('cbam' in exp for exp in RUN_EXPERIMENTS)
selected_requires_ghost = any('ghost' in exp for exp in RUN_EXPERIMENTS)
if selected_requires_cbam and not CBAM_YAML.exists():
    raise FileNotFoundError(f'Selected CBAM experiment but missing config: {CBAM_YAML}')
if selected_requires_ghost and not GHOST_YAML.exists():
    raise FileNotFoundError(f'Selected Ghost experiment but missing config: {GHOST_YAML}')

## 9. Debug and Oversampling Manifests

For oversampling, the notebook creates reproducible image-list manifests in `/kaggle/working/manifests`. It does not duplicate or modify raw image files. Validation and test splits remain unchanged.

In [ ]:
def label_for_image(img_path: Path, dataset_root: Path, split='train'):
    return dataset_root / 'labels' / split / f'{img_path.stem}.txt'

def image_contains_any_class(img_path: Path, dataset_root: Path, class_ids, split='train'):
    classes = parse_label_classes(label_for_image(img_path, dataset_root, split=split))
    return bool(set(classes).intersection(set(class_ids)))

def write_image_manifest(path: Path, images):
    path.write_text('\n'.join(str(Path(p).resolve()) for p in images) + '\n')
    return path

def create_train_manifest(dataset_root: Path, oversample=False, debug=False):
    train_imgs = image_files(dataset_root / 'images' / 'train')
    rng = random.Random(SEED)
    if debug:
        rng.shuffle(train_imgs)
        train_imgs = train_imgs[:DEBUG_MAX_TRAIN_IMAGES]
    if oversample:
        violation_imgs = [img for img in train_imgs if image_contains_any_class(img, dataset_root, VIOLATION_CLASS_IDS, split='train')]
        expanded = list(train_imgs) + violation_imgs * max(0, OVERSAMPLE_FACTOR - 1)
        rng.shuffle(expanded)
        name = f'train_oversample_x{OVERSAMPLE_FACTOR}' + ('_debug' if debug else '') + '.txt'
        path = MANIFESTS_DIR / name
        write_image_manifest(path, expanded)
        print(f'Oversample manifest: {path} total={len(expanded)} base={len(train_imgs)} violation={len(violation_imgs)}')
        return path
    name = 'train_debug.txt' if debug else None
    if debug:
        path = MANIFESTS_DIR / name
        write_image_manifest(path, train_imgs)
        print(f'Debug train manifest: {path} total={len(train_imgs)}')
        return path
    return 'images/train'

def create_eval_manifest(dataset_root: Path, split: str, debug=False):
    if not debug:
        return f'images/{split}'
    imgs = image_files(dataset_root / 'images' / split)
    split_offsets = {'train': 11, 'val': 17, 'test': 23}
    rng = random.Random(SEED + split_offsets.get(split, 0))
    rng.shuffle(imgs)
    imgs = imgs[:DEBUG_MAX_EVAL_IMAGES]
    path = MANIFESTS_DIR / f'{split}_debug.txt'
    write_image_manifest(path, imgs)
    print(f'Debug {split} manifest: {path} total={len(imgs)}')
    return path

def make_experiment_data_yaml(exp_name: str, oversample=False, debug=False):
    train_value = create_train_manifest(DATASET_ROOT, oversample=oversample, debug=debug)
    val_value = create_eval_manifest(DATASET_ROOT, 'val', debug=debug)
    test_value = create_eval_manifest(DATASET_ROOT, 'test', debug=debug)
    yaml_path = CONFIGS_DIR / f'data_{exp_name}.yaml'
    return write_data_yaml(yaml_path, DATASET_ROOT, str(train_value), str(val_value), str(test_value))

## 10. Experiment Registry

Each experiment has one explicit model source, one set of hyperparameters, and one output run name. The baseline behavior is preserved: `baseline_yolo26n` uses pretrained `yolo26n.pt` and the same hyperparameters as `train_baseline.py`.

In [ ]:
BASE_HYPERPARAMS = dict(
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    cos_lr=True,
    label_smoothing=0.1,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    degrees=10.0,
    translate=0.2,
    scale=0.5,
    fliplr=0.5,
)

# Same architecture as baseline, tuned only through augmentation/hyperparameters.
VIOLATION_AUG_HYPERPARAMS = dict(BASE_HYPERPARAMS)
VIOLATION_AUG_HYPERPARAMS.update(
    mosaic=1.0,
    mixup=0.25,
    copy_paste=0.25,
    scale=0.65,
    translate=0.25,
)

EXPERIMENTS = {
    'baseline_yolo26n': dict(model='yolo26n.pt', hparams=BASE_HYPERPARAMS, oversample=False, target='baseline'),
    'baseline_violation_aug': dict(model='yolo26n.pt', hparams=VIOLATION_AUG_HYPERPARAMS, oversample=False, target='imbalance_aug'),
    'baseline_violation_oversample': dict(model='yolo26n.pt', hparams=BASE_HYPERPARAMS, oversample=True, target='imbalance_oversample'),
    'cbam_existing_repro': dict(model=str(CBAM_YAML), hparams=BASE_HYPERPARAMS, oversample=False, target='architecture'),
    'cbam_violation_oversample': dict(model=str(CBAM_YAML), hparams=BASE_HYPERPARAMS, oversample=True, target='architecture_plus_imbalance'),
    'ghost_existing_repro': dict(model=str(GHOST_YAML), hparams=BASE_HYPERPARAMS, oversample=False, target='architecture_efficiency'),
    'ghost_violation_oversample': dict(model=str(GHOST_YAML), hparams=BASE_HYPERPARAMS, oversample=True, target='architecture_plus_imbalance'),
}

unknown = [exp for exp in RUN_EXPERIMENTS if exp not in EXPERIMENTS]
if unknown:
    raise ValueError(f'Unknown experiments selected: {unknown}')

effective_epochs = DEBUG_EPOCHS if FAST_DEBUG else EPOCHS
effective_imgsz = DEBUG_IMGSZ if FAST_DEBUG else IMGSZ
effective_batch = min(BATCH, 16) if FAST_DEBUG else BATCH

print('effective_epochs:', effective_epochs)
print('effective_imgsz:', effective_imgsz)
print('effective_batch:', effective_batch)
print('selected:', RUN_EXPERIMENTS)

## 11. Ultralytics Setup Helpers

CBAM custom YAML loading may require registering `CBAM` into the Ultralytics task namespace, matching the existing Modal script workaround. GhostConv is normally available directly.

In [ ]:
from ultralytics import YOLO

def register_custom_modules_if_needed(model_source: str):
    model_source_lower = str(model_source).lower()
    if 'cbam' in model_source_lower:
        try:
            from ultralytics.nn.modules.conv import CBAM
            import ultralytics.nn.tasks as nn_tasks
            nn_tasks.CBAM = CBAM
            print('Registered CBAM from ultralytics.nn.modules.conv')
        except Exception as exc:
            try:
                from ultralytics.nn.modules.block import CBAM
                import ultralytics.nn.tasks as nn_tasks
                nn_tasks.CBAM = CBAM
                print('Registered CBAM from ultralytics.nn.modules.block')
            except Exception as inner_exc:
                raise ImportError(f'Could not register CBAM: {exc}; {inner_exc}')

def load_model(model_source: str):
    register_custom_modules_if_needed(model_source)
    return YOLO(model_source)

def save_json(path: Path, obj):
    path.write_text(json.dumps(obj, indent=2, default=str) + '\n')
    return path

## 12. Training Runner

Experiments run sequentially. A single experiment may use both T4 GPUs through `device='0,1'` when Kaggle exposes two CUDA devices.

In [ ]:
def experiment_run_dir(exp_name: str):
    return RUNS_DIR / exp_name

def best_weight_path(exp_name: str):
    return experiment_run_dir(exp_name) / 'weights' / 'best.pt'

def last_weight_path(exp_name: str):
    return experiment_run_dir(exp_name) / 'weights' / 'last.pt'

def train_experiment(exp_name: str):
    cfg = EXPERIMENTS[exp_name]
    model_source = cfg['model']
    data_yaml = make_experiment_data_yaml(exp_name, oversample=cfg['oversample'], debug=FAST_DEBUG)
    run_dir = experiment_run_dir(exp_name)
    hparams = dict(cfg['hparams'])
    train_config = dict(
        experiment=exp_name,
        target=cfg['target'],
        model=model_source,
        data=str(data_yaml),
        epochs=effective_epochs,
        imgsz=effective_imgsz,
        batch=effective_batch,
        device=DEVICE,
        seed=SEED,
        hparams=hparams,
        fast_debug=FAST_DEBUG,
        command_equivalent=f"YOLO({model_source!r}).train(data={str(data_yaml)!r}, epochs={effective_epochs}, imgsz={effective_imgsz}, batch={effective_batch}, device={DEVICE!r}, seed={SEED}, project={str(RUNS_DIR)!r}, name={exp_name!r}, hparams={hparams!r})",
    )
    save_json(LOGS_DIR / f'{exp_name}_train_config.json', train_config)
    print('\n' + '=' * 80)
    print('TRAIN:', exp_name)
    print(json.dumps(train_config, indent=2))
    print('=' * 80)

    if DEVICE == 'cpu' and not FAST_DEBUG:
        raise RuntimeError('CUDA is unavailable. Set FAST_DEBUG=True for CPU smoke checks or enable a Kaggle GPU accelerator.')

    model = load_model(model_source)
    results = model.train(
        data=str(data_yaml),
        epochs=effective_epochs,
        imgsz=effective_imgsz,
        batch=effective_batch,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=exp_name,
        exist_ok=True,
        seed=SEED,
        plots=True,
        save=True,
        save_period=10,
        pretrained=True if model_source == 'yolo26n.pt' else cfg['target'] != 'architecture_efficiency',
        **hparams,
    )
    return results

trained = {}
if RUN_TRAINING:
    for exp_name in RUN_EXPERIMENTS:
        trained[exp_name] = str(train_experiment(exp_name))
else:
    print('RUN_TRAINING=False. Skipping training and using existing weights if present.')

## 13. Violation-Focused Evaluation

This evaluation uses the same test split for every experiment and records overall metrics plus per-class AP for violation classes.

In [ ]:
def safe_float(value, default=None):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default

def evaluate_experiment(exp_name: str):
    cfg = EXPERIMENTS[exp_name]
    weights = best_weight_path(exp_name)
    if not weights.exists():
        weights = last_weight_path(exp_name)
    if not weights.exists():
        print(f'Skipping evaluation for {exp_name}: no weights found')
        return dict(experiment=exp_name, status='missing_weights')

    data_yaml = make_experiment_data_yaml(exp_name + '_eval', oversample=False, debug=FAST_DEBUG)
    eval_config = dict(experiment=exp_name, weights=str(weights), data=str(data_yaml), device=DEVICE, imgsz=effective_imgsz)
    save_json(LOGS_DIR / f'{exp_name}_eval_config.json', eval_config)

    print('\n' + '=' * 80)
    print('EVAL:', exp_name)
    print(json.dumps(eval_config, indent=2))
    print('=' * 80)

    model = load_model(str(weights))
    metrics = model.val(
        data=str(data_yaml),
        split='test',
        imgsz=effective_imgsz,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=f'{exp_name}_test',
        exist_ok=True,
    )

    per_class_ap50 = getattr(metrics.box, 'ap50', [])
    violation_scores = {}
    for cls_id in VIOLATION_CLASS_IDS:
        value = per_class_ap50[cls_id] if cls_id < len(per_class_ap50) else None
        violation_scores[f'{CLASS_NAMES[cls_id]}_map50'] = safe_float(value)
    valid_violation = [v for v in violation_scores.values() if v is not None]
    speed = getattr(metrics, 'speed', {}) or {}
    result = dict(
        experiment=exp_name,
        status='ok',
        target=cfg['target'],
        weights=str(weights),
        map50=safe_float(getattr(metrics.box, 'map50', None)),
        map50_95=safe_float(getattr(metrics.box, 'map', None)),
        precision=safe_float(getattr(metrics.box, 'mp', None)),
        recall=safe_float(getattr(metrics.box, 'mr', None)),
        inference_ms=safe_float(speed.get('inference')),
        mean_violation_map50=safe_float(np.mean(valid_violation)) if valid_violation else None,
    )
    result.update(violation_scores)
    print(json.dumps(result, indent=2))
    return result

evaluation_results = []
if RUN_EVALUATION:
    for exp_name in RUN_EXPERIMENTS:
        evaluation_results.append(evaluate_experiment(exp_name))
else:
    print('RUN_EVALUATION=False. Skipping evaluation.')

## 14. Save Comparison Tables

The summary is saved as both CSV and JSON under `/kaggle/working/results`.

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(evaluation_results)
summary_csv = RESULTS_DIR / 'experiment_summary.csv'
summary_json = RESULTS_DIR / 'experiment_summary.json'
summary_df.to_csv(summary_csv, index=False)
summary_json.write_text(summary_df.to_json(orient='records', indent=2) + '\n')

print('Saved:', summary_csv)
print('Saved:', summary_json)
display(summary_df)

## 15. Select the Best Model

By default, the best model is selected by `mean_violation_map50`. Change `BEST_MODEL_SELECTION_METRIC` near the top if you want to select by another metric such as overall `map50`.

In [ ]:
def select_best_model(df: pd.DataFrame, metric: str):
    if df.empty:
        print('No evaluation results available.')
        return None
    ok = df[df.get('status') == 'ok'].copy()
    if ok.empty:
        print('No successful evaluations available.')
        return None
    if metric not in ok.columns:
        raise KeyError(f'Selection metric not found: {metric}. Available columns: {list(ok.columns)}')
    ok[metric] = pd.to_numeric(ok[metric], errors='coerce')
    ok = ok.dropna(subset=[metric])
    if ok.empty:
        print(f'No non-null values for metric {metric}.')
        return None
    row = ok.sort_values(metric, ascending=False).iloc[0].to_dict()
    return row

best_result = select_best_model(summary_df, BEST_MODEL_SELECTION_METRIC)
save_json(RESULTS_DIR / 'best_model_selection.json', best_result)
print('Selection metric:', BEST_MODEL_SELECTION_METRIC)
print(json.dumps(best_result, indent=2, default=str))

## 16. Export Best Model to ONNX

The Modal `export_model.py` script is not used directly on Kaggle because it assumes Modal volumes. This cell performs the equivalent Ultralytics ONNX export and copies exported models to `/kaggle/working/exports`.

In [ ]:
def export_best_model(best):
    if not best:
        print('No best model selected; skipping export.')
        return None
    weights = Path(best['weights'])
    if not weights.exists():
        print(f'Best weights not found: {weights}')
        return None
    exp_name = best['experiment']
    export_config = dict(experiment=exp_name, weights=str(weights), imgsz=effective_imgsz, format='onnx')
    save_json(LOGS_DIR / f'{exp_name}_export_config.json', export_config)
    print('\n' + '=' * 80)
    print('EXPORT BEST:', exp_name)
    print(json.dumps(export_config, indent=2))
    print('=' * 80)

    model = load_model(str(weights))
    exported_path = model.export(format='onnx', imgsz=effective_imgsz, simplify=True)
    exported_path = Path(exported_path)
    dest = EXPORTS_DIR / f'{exp_name}_best.onnx'
    shutil.copy2(exported_path, dest)
    info = dict(experiment=exp_name, source=str(exported_path), exported=str(dest), size_mb=dest.stat().st_size / (1024 * 1024))
    save_json(RESULTS_DIR / 'export_summary.json', info)
    print(json.dumps(info, indent=2))
    return info

export_info = None
if RUN_EXPORT:
    export_info = export_best_model(best_result)
else:
    print('RUN_EXPORT=False. Skipping export.')

## 17. Package Useful Outputs

This collects logs, config snapshots, summaries, exports, and best weights under `/kaggle/working`. Kaggle automatically preserves `/kaggle/working` as notebook output.

In [ ]:
PACKAGE_DIR = KAGGLE_WORKING / 'ppe_experiment_package'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

def copy_dir_contents(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        return
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)

for src, name in [
    (RESULTS_DIR, 'results'),
    (EXPORTS_DIR, 'exports'),
    (LOGS_DIR, 'logs'),
    (CONFIGS_DIR, 'configs'),
    (MANIFESTS_DIR, 'manifests'),
]:
    copy_dir_contents(src, PACKAGE_DIR / name)

weights_dir = PACKAGE_DIR / 'weights'
weights_dir.mkdir(exist_ok=True)
for exp_name in RUN_EXPERIMENTS:
    for candidate_name in ['best.pt', 'last.pt']:
        candidate = experiment_run_dir(exp_name) / 'weights' / candidate_name
        if candidate.exists():
            shutil.copy2(candidate, weights_dir / f'{exp_name}_{candidate_name}')

run_metadata = dict(
    created_at=datetime.utcnow().isoformat() + 'Z',
    dataset_root=str(DATASET_ROOT),
    project_root=str(PROJECT_ROOT),
    run_experiments=RUN_EXPERIMENTS,
    fast_debug=FAST_DEBUG,
    device=DEVICE,
    seed=SEED,
    best_model_selection_metric=BEST_MODEL_SELECTION_METRIC,
)
save_json(PACKAGE_DIR / 'run_metadata.json', run_metadata)

print('Packaged outputs under:', PACKAGE_DIR)
list_tree(PACKAGE_DIR, max_depth=2)

## 18. Notes for Full Runs

- Start with `FAST_DEBUG=True`, `DEBUG_EPOCHS=1`, and a small `RUN_EXPERIMENTS` list to verify paths and model construction.
- For full training, set `FAST_DEBUG=False` and keep experiments sequential. Each selected experiment may use both T4 GPUs through `DEVICE='0,1'`.
- To skip training and evaluate/export existing weights, set `RUN_TRAINING=False` and keep the run directories under `/kaggle/working/ppe_yolo_experiments/runs`.
- The primary selection metric is `mean_violation_map50`, which averages mAP@0.5 for `no_helmet`, `no_goggle`, `no_gloves`, and `no_boots`.